# DSPy: Programming Instead of Prompting

In this notebook, we'll implement a basic DSPy pipeline to demonstrate how to optimize prompts automatically using **Signatures** and **BootstrapFewShot**.

In [ ]:
# !pip install dspy-ai openai

## 1. Defining a Signature

A Signature is a declarative contract. No more "You are a helpful assistant...".

In [ ]:
import dspy

class SentimentAnalysis(dspy.Signature):
    """Classify the sentiment of a sentence as Positive, Negative, or Neutral."""
    sentence = dspy.InputField()
    sentiment = dspy.OutputField(desc="One of: Positive, Negative, Neutral")

# 2. Creating a Module (Chain of Thought)
class SentimentClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(SentimentAnalysis)
    
    def forward(self, sentence):
        return self.prog(sentence=sentence)

## 2. Setting up the Optimizer

The power of DSPy lies in automatically selecting the best Few-shot examples (Bootstrapping).

In [ ]:
from dspy.teleprompt import BootstrapFewShot

# Example Training Data (Small Golden Set)
trainset = [
    dspy.Example(sentence="I love this app!", sentiment="Positive").with_inputs('sentence'),
    dspy.Example(sentence="This is garbage.", sentiment="Negative").with_inputs('sentence'),
    dspy.Example(sentence="It works as expected.", sentiment="Neutral").with_inputs('sentence')
]

# Metric: Is the output exactly equal to the label?
def validate_sentiment(example, pred, trace=None):
    return example.sentiment.lower() == pred.sentiment.lower()

# 3. Compile the Program
# This will automatically pick examples and refine the prompt
config = dspy.settings.configure(lm=dspy.OpenAI(model='gpt-4o-mini', api_key='sk-...'))
teleprompter = BootstrapFewShot(metric=validate_sentiment)
# compiled_classifier = teleprompter.compile(SentimentClassifier(), trainset=trainset)

print("DSPy Pipeline defined: Signature -> Module -> Optimizer.")

## Why this is Industrial Standard:

1. **Model Agnostic**: Change the LM in `dspy.settings`, and the optimizer will find new best examples for THAT model.
2. **Metric Driven**: We optimize based on a function (`validate_sentiment`), not vibes.
3. **Modular**: We can stack modules like `ChainOfThought` inside `ProgramOfThought` effortlessly.